In [5]:
# import sys
# from pathlib import Path

# #add the parent directory to python's module search path
# sys.path.append(str(Path(__file__).parent.parent))

In [7]:
import sys
from pathlib import Path

#get current working directoy
parent_dir = sys.path.append(str(Path().cwd().parent))



In [12]:
from langchain_pinecone import PineconeVectorStore
from src.helpers import load_hugging_face_embeddings
from pinecone import Pinecone
import os
from langchain.tools import tool

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)
print(PINECONE_API_KEY)


None
pcsk_52fsWW_PyGokwzc1GSCGMNuJ5rEueegbVL8sXShu6MSvcivuUM17LvrtJwLS3Kn2ffUntz


## RAW_RAG_RESPONSE

In [26]:
index_name = "travel-guru"

@tool
def search_pdf_docs(query:str) -> str:
    """Query pdf documents indexed in Pinecone"""
    embeddings = load_hugging_face_embeddings()
    vectorstore = PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )
    docs = retriever.get_relevant_documents(query)
    return "\n\n".join([d.page_content for d in docs]) or "No results found"


query = "What are the routes to the Everest Base Camp?"
raw_rag_response = search_pdf_docs.invoke({"query": query})
print(raw_rag_response)

e x p e r i e n c e . 
E v e r e s t B a s e C a m p T r e k 
Everest Base Camp T rek Itinerary 
D a y 1 : F l i g h t f r o m K a t h m a n d u t o L u k l a a n d T r e k t o P h a k d i n g 
2 4

TRIP FACTSEVEREST BASE CAMP TREK
Everest Base Camp Trek is the busiest trail in 
Khumbu and, after a short ﬂight from 
Kathmandu, you can start your trek. A ﬁrst 
acclimatization day is normally set at Namche, 
during which you can explore the Sherpa 
Museum, Syangboche Airport, Everest View 
Hotel and Edmund Hillary Memorial, as well as 
the historic twin villages of Khumjung and 
Khunde. There is a dramatic rise in altitude, so it 
is recommended to have at least two

Thamel, Kathmandu,
Nepal P .O.Box: 4003
+977-9841773981
+977-9851139218
info@magicalnepal.com
www.magicalnepal.com
1 WEEK TREK
› Everest Panorama
› Trek
 Helambu Trek
› Ghorepani Poonhill Trek
2 WEEK TREK
› Gokyo Lake Trek
› Everest Base Camp Trek
› Langtang Trek
› Gosaikunda 
Lake Trek
› Tamang Heritage Trek
› Lower Manaslu

## COMBINE RAG RESPONSE WITH LLM

In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from src.prompt import *
from langchain.chains import LLMChain

llm = ChatGoogleGenerativeAI(
    google_api_key=os.getenv("GEMINI_API_KEY"),
    model="gemini-2.0-flash",
    temperature=0.2,
    max_tokens=1024,
    max_retries=3
)

PROMPT = PromptTemplate(
    template=system_prompt,
    input_variables=["context","question"]
)

qa_chain = LLMChain(llm=llm, prompt=PROMPT)
query = "What are the routes to the Everest Base Camp?"
raw_rag_response = search_pdf_docs.invoke({"query": query})

result = qa_chain.run({
    "context": raw_rag_response,
    "question": query
})

print("Final Response: ", result)

Final Response:  Understood. I'm ready to answer your questions about trekking in Nepal! Ask me anything about routes, permits, gear, planning, and logistics.
